# Image downloading for collection items

- The following script is used in order to download the images of folk paintings and icons at the Krovets Collection

- The script uses the **Europeana** API endpoint to fetch all the necessary meta-data for the items in the collection (https://api.europeana.eu/set/26106.json) and then, using their ids and image urls, the images are downloaded inside the user's drive folder.

Connect to Google Drive folders (a pop-up window may appear) and use API key from secrets.

In order to use the API you should get an API key by creating an account on **Europeana**. After getting your key, paste it and save it on secrets (key icon on the left side). The keys should only be visible to you.

In [ ]:
import requests
import pandas as pd
from google.colab import userdata
from google.colab import drive

drive.mount('/content/drive')

EUROPEANA_API_KEY = userdata.get('EUROPEANA_API_KEY')

Mounted at /content/drive


Folk paintings and icons are fetched from the collection API endpoint. Then, their ids are used for image downloading into the respective **Drive folder**.

- As suggested by Europeana, API keys are included in the request inside the **headers** and not as a parameter. Make sure to take this into account if you wish to recreate this research by using the API

In [ ]:
headers = {
    'X-API-KEY': EUROPEANA_API_KEY
}

url = "https://api.europeana.eu/set/26106.json"

ids = []
page = 1
page_size = 100

while True:
    params = {
        'profile': 'items',
        'page': page,
        'pageSize': page_size
    }

    response = requests.get(url, params=params, headers=headers, timeout=60)
    response.raise_for_status() # Raise an HTTPError for bad responses
    data = response.json()
    items = data.get('items', [])

    if not items:
        break

    ids.extend([uri.split('/')[-1] for uri in items if isinstance(uri, str)])
    print(f"Fetched {len(ids)} items so far (page {page})")

    if 'next' not in data:
        break
    page += 1

print(f"Successfully fetched metadata for {len(ids)} items.")

Total items available in the collection: 312
Fetched 100 items so far (page 1/4)
Fetched 200 items so far (page 2/4)
Fetched 300 items so far (page 3/4)
Fetched 312 items so far (page 4/4)
Successfully fetched metadata for 312 items.


Confirmation that the items we wanted were fetched successfully

In [ ]:
print(ids[:5])

['KYD1241', 'KYD1242', 'KYD1243', 'KYD1244', 'KYD1245']


Using **multi-threading**, the following part once again makes requests to the API to get further detailed data on the records we want. In that way we can have the urls of their **images**.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

item_to_image_url = {}

# Given an item's id, the function will try to fetch the item's image using its URL
def fetch_single_item(item_id):
  record_url = f"https://api.europeana.eu/record/v2/1413/{item_id}.json"
  params = {
      "profile": "rich",
  }
  try:
    response = requests.get(url=record_url, params=params, headers=headers, timeout=60)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx). If there is such error, request fails.
    data = response.json()
    obj = data.get('object') or {}

    image_url = None

    # Keep only original provider image URL
    for agg in (obj.get('aggregations') or []):
      candidate = agg.get('edmIsShownBy')
      if isinstance(candidate, str) and candidate.startswith('http'):
        image_url = candidate
        break

    return {item_id: image_url} if image_url else None
  except requests.exceptions.RequestException as e: # if the URL does not match a valid image
    print(f"Error fetching item {item_id}: {e}")
    return None

# Using ThreadPoolExecutor to make requests concurrently
# max_workers can be tuned based on network conditions and API limits
with ThreadPoolExecutor(max_workers=20) as executor:
    # Submit all tasks and get Future objects
    future_to_id = {executor.submit(fetch_single_item, item_id): item_id for item_id in ids}

    for i, future in enumerate(future_to_id):
        item_id = future_to_id[future]
        try:
            item_result = future.result()
            if item_result: # Only append if item_result is not None
                item_to_image_url.update(item_result)
        except Exception as exc:
            print(f'{item_id} generated an exception: {exc}')
        finally:
            if (i+1) % 8 == 0:
              print(f"Fetched {i+1}/{len(ids)} items.")

print(f"Successfully fetched data for {len(item_to_image_url)} records.")

Fetched 8/312 items.
Fetched 16/312 items.
Fetched 24/312 items.
Fetched 32/312 items.
Fetched 40/312 items.
Fetched 48/312 items.
Fetched 56/312 items.
Fetched 64/312 items.
Fetched 72/312 items.
Fetched 80/312 items.
Fetched 88/312 items.
Fetched 96/312 items.
Fetched 104/312 items.
Fetched 112/312 items.
Fetched 120/312 items.
Fetched 128/312 items.
Fetched 136/312 items.
Fetched 144/312 items.
Fetched 152/312 items.
Fetched 160/312 items.
Fetched 168/312 items.
Fetched 176/312 items.
Fetched 184/312 items.
Fetched 192/312 items.
Fetched 200/312 items.
Fetched 208/312 items.
Fetched 216/312 items.
Fetched 224/312 items.
Fetched 232/312 items.
Fetched 240/312 items.
Fetched 248/312 items.
Fetched 256/312 items.
Fetched 264/312 items.
Fetched 272/312 items.
Fetched 280/312 items.
Fetched 288/312 items.
Fetched 296/312 items.
Fetched 304/312 items.
Fetched 312/312 items.
Successfully fetched data for 312 records.


Save all item ids and their respective urls to a dataframe

In [ ]:
image_df = pd.DataFrame(item_to_image_url.items(), columns=["id", "image_url"])
print(image_df.head())

              id                                          image_url
0  /1413/KYD1241  https://krovets.ua/disk/products/LMYFJXHaqKTZQ...
1  /1413/KYD1242  https://krovets.ua/disk/products/JQhNZnEgo1HYe...
2  /1413/KYD1243  https://krovets.ua/disk/products/ZTEI5hDDqRPcK...
3  /1413/KYD1244  https://krovets.ua/disk/products/INVENOJP5qtuz...
4  /1413/KYD1245  https://krovets.ua/disk/products/XOTfJyzBaAqwB...


Download all images from their saved urls. It may take up to 30 seconds to download all the images.

In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import threading

# Define the directory to save images in Google Drive
save_directory = '/content/drive/My Drive/Folk-paintings-and-icons'

# Create the directory if it doesn't exist
os.makedirs(save_directory, exist_ok=True)
print(f"Save directory created: {save_directory}")

# Counter for successfully downloaded images
downloaded_count = 0

print(f"Attempting to download {len(image_df['id'])} images...")

# Use a Lock for thread-safe increment of downloaded_count
_download_lock = threading.Lock()

def download_single_image(item_id, url, save_dir):
    global downloaded_count # Declare as global to modify the counter
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

        filename = f"{item_id}.jpg"
        file_path = os.path.join(save_dir, filename)

        # Convert to RGB to avoid JPEG save errors on RGBA/P images
        image = Image.open(BytesIO(response.content)).convert('RGB')
        image.save(file_path, 'jpeg') # Saving as jpeg as per previous logic and consistent with filename

        # Safely increment downloaded_count for progress tracking
        with _download_lock: # Use a lock for thread-safe increment
            downloaded_count += 1
            if downloaded_count % 100 == 0:
                print(f"Downloaded {downloaded_count} images so far...")
        return True

    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return False
    except Exception as e:
        print(f"Error processing image from {url}: {e}")
        return False


# Using ThreadPoolExecutor to download images concurrently
# max_workers can be tuned based on network conditions and local resources
# More workers may increase downloading speed. However, it is not suggested under limited network circumstances.
# We suggest 20 workers for downloading 312 images, as it takes ~7 seconds
with ThreadPoolExecutor(max_workers=20) as executor:
    # Submit all download tasks
    # Pass row['id'] (which is item_id_full) to the function
    futures = [executor.submit(download_single_image, row['id'], row['image_url'], save_directory)
               for _, row in image_df.iterrows()]

    # Wait for all futures to complete
    # The progress printing inside download_single_image will handle updates
    for future in futures:
        future.result() # This will re-raise any exceptions that occurred in the thread

print(f"Successfully downloaded {downloaded_count} images to {save_directory}")

Save directory created: /content/drive/My Drive/Folk-paintings-and-icons
Attempting to download 312 images...
Downloaded 100 images so far...
Downloaded 200 images so far...
Downloaded 300 images so far...
Successfully downloaded 312 images to /content/drive/My Drive/Folk-paintings-and-icons
